In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

In [20]:
# Input
INPUT_PATH = Path("../data/raw/DataResidentsbyPlngAreaSubzoneJun24.csv")

# Output
OUTPUT_PATH = Path("../data/processed/demand_surface.csv")

# WiSE 2023 prevalence rates by age band
RATE_60_74 = 0.030   # 3.0%
RATE_75_84 = 0.182   # 18.2%
RATE_85_PLUS = 0.486 # 48.6%

In [21]:
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")

print("Shape:", df.shape)
print(df.head())

Shape: (100928, 7)
           PA                      SZ      AG    Sex  \
0  Ang Mo Kio  Ang Mo Kio Town Centre  0_to_4  Males   
1  Ang Mo Kio  Ang Mo Kio Town Centre  0_to_4  Males   
2  Ang Mo Kio  Ang Mo Kio Town Centre  0_to_4  Males   
3  Ang Mo Kio  Ang Mo Kio Town Centre  0_to_4  Males   
4  Ang Mo Kio  Ang Mo Kio Town Centre  0_to_4  Males   

                                       TOD  Pop  Time  
0                  HDB 1- and 2-Room Flats    0  2024  
1                         HDB 3-Room Flats    0  2024  
2                         HDB 4-Room Flats   10  2024  
3           HDB 5-Room and Executive Flats   20  2024  
4  HUDC Flats (excluding those privatised)    0  2024  


In [22]:
print("Age groups:", sorted(df["AG"].unique()))
print()
print("Sex values:", df["Sex"].unique())
print()
print("TOD values:", df["TOD"].unique())
print()
print("Total planning areas:", df["PA"].nunique())

Age groups: ['0_to_4', '10_to_14', '15_to_19', '20_to_24', '25_to_29', '30_to_34', '35_to_39', '40_to_44', '45_to_49', '50_to_54', '55_to_59', '5_to_9', '60_to_64', '65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', '90_and_over']

Sex values: ['Males' 'Females']

TOD values: ['HDB 1- and 2-Room Flats' 'HDB 3-Room Flats' 'HDB 4-Room Flats'
 'HDB 5-Room and Executive Flats'
 'HUDC Flats (excluding those privatised)'
 'Condominiums and Other Apartments' 'Landed Properties' 'Others']

Total planning areas: 55


In [9]:
age_groups_needed = [
    "60_to_64", "65_to_69", "70_to_74",   # maps to 60–74 band
    "75_to_79", "80_to_84",                # maps to 75–84 band
    "85_to_89", "90_and_over"              # maps to 85+ band
]

df_aged = df[df["AG"].isin(age_groups_needed)].copy()

print("Rows after age filter:", df_aged.shape[0])
print("Unique age groups remaining:", sorted(df_aged["AG"].unique()))

Rows after age filter: 37184
Unique age groups remaining: ['60_to_64', '65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', '90_and_over']


In [10]:
conditions = [
    df_aged["AG"].isin(["60_to_64", "65_to_69", "70_to_74"]),
    df_aged["AG"].isin(["75_to_79", "80_to_84"]),
    df_aged["AG"].isin(["85_to_89", "90_and_over"])
]

labels = ["60_74", "75_84", "85_plus"]

df_aged["band"] = np.select(conditions, labels, default="unknown")

print(df_aged["band"].value_counts())
print()
print("Unknown band rows:", (df_aged["band"] == "unknown").sum())

band
60_74      15936
75_84      10624
85_plus    10624
Name: count, dtype: int64

Unknown band rows: 0


In [11]:
df_band_pop = (
    df_aged
    .groupby(["PA", "band"])["Pop"]
    .sum()
    .reset_index()
)

print("Shape:", df_band_pop.shape)
print(df_band_pop.head(10))

Shape: (165, 3)
           PA     band    Pop
0  Ang Mo Kio    60_74  35520
1  Ang Mo Kio    75_84  13080
2  Ang Mo Kio  85_plus   4240
3       Bedok    60_74  59230
4       Bedok    75_84  18960
5       Bedok  85_plus   6110
6      Bishan    60_74  18510
7      Bishan    75_84   5560
8      Bishan  85_plus   1980
9    Boon Lay    60_74      0


In [12]:
df_wide = (
    df_band_pop
    .pivot(index="PA", columns="band", values="Pop")
    .reset_index()
)

df_wide.columns.name = None  # Remove the "band" label from the column axis

df_wide = df_wide.rename(columns={
    "60_74": "pop_60_74",
    "75_84": "pop_75_84",
    "85_plus": "pop_85_plus"
})

print(df_wide.shape)
print(df_wide.head())

(55, 4)
            PA  pop_60_74  pop_75_84  pop_85_plus
0   Ang Mo Kio      35520      13080         4240
1        Bedok      59230      18960         6110
2       Bishan      18510       5560         1980
3     Boon Lay          0          0            0
4  Bukit Batok      32060       7250         2130


In [13]:
df_wide["pwd_60_74"]   = df_wide["pop_60_74"]   * RATE_60_74
df_wide["pwd_75_84"]   = df_wide["pop_75_84"]   * RATE_75_84
df_wide["pwd_85_plus"] = df_wide["pop_85_plus"] * RATE_85_PLUS

df_wide["pwd_estimate"] = (
    df_wide["pwd_60_74"] +
    df_wide["pwd_75_84"] +
    df_wide["pwd_85_plus"]
).round().astype(int)

print(df_wide[["PA", "pop_60_74", "pop_75_84", "pop_85_plus", "pwd_estimate"]].head(10))
print()
print("Total estimated PWDs across Singapore:", df_wide["pwd_estimate"].sum())

                        PA  pop_60_74  pop_75_84  pop_85_plus  pwd_estimate
0               Ang Mo Kio      35520      13080         4240          5507
1                    Bedok      59230      18960         6110          8197
2                   Bishan      18510       5560         1980          2530
3                 Boon Lay          0          0            0             0
4              Bukit Batok      32060       7250         2130          3316
5              Bukit Merah      30240      12140         4520          5313
6            Bukit Panjang      26730       5810         1760          2715
7              Bukit Timah      13400       4810         1930          2215
8  Central Water Catchment          0          0            0             0
9                   Changi        220        100           40            44

Total estimated PWDs across Singapore: 95647


In [14]:
df_wide["PLN_AREA_N"] = df_wide["PA"].str.upper()

print(df_wide[["PA", "PLN_AREA_N"]].head())

            PA   PLN_AREA_N
0   Ang Mo Kio   ANG MO KIO
1        Bedok        BEDOK
2       Bishan       BISHAN
3     Boon Lay     BOON LAY
4  Bukit Batok  BUKIT BATOK


In [15]:
df_output = (
    df_wide[["PLN_AREA_N", "pwd_estimate"]]
    .sort_values("PLN_AREA_N")
    .reset_index(drop=True)
)

print(df_output)
print()
print("Row count:", len(df_output))
print("Total estimated PWDs:", df_output["pwd_estimate"].sum())

                 PLN_AREA_N  pwd_estimate
0                ANG MO KIO          5507
1                     BEDOK          8197
2                    BISHAN          2530
3                  BOON LAY             0
4               BUKIT BATOK          3316
5               BUKIT MERAH          5313
6             BUKIT PANJANG          2715
7               BUKIT TIMAH          2215
8   CENTRAL WATER CATCHMENT             0
9                    CHANGI            44
10               CHANGI BAY             0
11            CHOA CHU KANG          3036
12                 CLEMENTI          2900
13            DOWNTOWN CORE            60
14                  GEYLANG          3398
15                  HOUGANG          5509
16              JURONG EAST          2101
17              JURONG WEST          4691
18                  KALLANG          3338
19             LIM CHU KANG             0
20                   MANDAI            44
21              MARINA EAST             0
22             MARINA SOUTH       

## Methodology Note — PWD Estimate vs WiSE 2023 Benchmark

### Figures

| Source | 60+ Population Base | Estimated PWDs |
|---|---|---|
| WiSE 2023 (Subramaniam et al., 2025) | ~840,000 (implied: 73,918 ÷ 8.8%) | 73,918 |
| SingStat Population in Brief 2023 | ~900,000 (citizens + PRs) | ~81,685 |
| This notebook (SingStat June 2024) | 1,053,850 (citizens + PRs) | 95,649 |

Age-band breakdown (this notebook, 2024 base):

| Age Band | Population | Prevalence Rate | Estimated PWDs |
|---|---|---|---|
| 60–74 | 771,260 | 3.0% | 23,138 |
| 75–84 | 213,250 | 18.2% | 38,812 |
| 85+ | 69,340 | 48.6% | 33,699 |
| **Total** | **1,053,850** | **9.08% blended** | **95,649** |

### Why this notebook produces a higher figure than WiSE 2023

Two factors explain the 21,731 difference — neither is a methodology error.

**Factor 1 — Population definition.** WiSE's implied denominator of ~840,000
is lower than SingStat's ~900,000 for the same period. WiSE surveyed
community-dwelling elderly and likely anchored to a citizens-only or
2022 reference population. SingStat counts all residents (citizens +
permanent residents). This accounts for roughly 60,000 of the base
population difference.

**Factor 2 — Reference year.** WiSE's survey ran March 2022 to September
2023. This notebook uses June 2024 SingStat data. Singapore's 60+ cohort
is growing rapidly as the early 1960s birth cohort ages in. The
additional ~154,000 elderly residents between 2023 and 2024 mechanically
produce more estimated PWDs at the same prevalence rates.

**Blended effective prevalence rate: 9.08%** — slightly above WiSE's
headline 8.8% because the 2024 population has a larger share of older
elderly (75–84 and 85+), who carry higher prevalence rates. This is
expected and consistent with Singapore's ageing trajectory.

### Appropriate use of this estimate

This notebook's figure of 95,649 is the correct basis for this analysis.
It represents estimated PWDs among all Singapore residents aged 60+ as
of June 2024, using the most current available prevalence rates (WiSE
2023) and the most current available population data (SingStat June
2024). The WiSE 73,918 figure is the 2022–2023 benchmark, not a ceiling
this estimate should be constrained to match.

In [16]:
df_output.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows written: {len(df_output)}")

Saved: ..\data\processed\demand_surface.csv
Rows written: 55
